<a href="https://colab.research.google.com/github/GARVIT-bit/ai_LAB/blob/main/FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### FOL Resolution Algorithm Implementation
Below is a general propositional/FOL-style resolution solver implementation in Python that resolves a set of clauses to check for a contradiction (empty clause).

In [3]:
def resolve(ci, cj):
    """Resolves two clauses ci and cj (represented as sets of literals)."""
    resolvents = []
    for di in ci:
        for dj in cj:
            # If one literal is the negation of the other
            if di == ('~' + dj) or ('~' + di) == dj:
                # Form the union of both clauses excluding the complementary literals
                new_clause = (ci - {di}) | (cj - {dj})
                if new_clause not in resolvents:
                    resolvents.append(new_clause)
    return resolvents

def pl_resolution(KB, query):
    """Propositional logic resolution algorithm.
    Returns True if KB entails query (i.e., KB ^ ~query is unsatisfiable), False otherwise.
    """
    # Add negated query to KB clauses
    clauses = [set(c) for c in KB] + [{ '~' + query if not query.startswith('~') else query[1:] }]

    print("Initial Clauses:")
    for idx, c in enumerate(clauses):
        print(f" C{idx+1}: {c if c else 'EMPTY'}")

    new = set()
    step = 1
    while True:
        n = len(clauses)
        pairs = [(clauses[i], clauses[j]) for i in range(n) for j in range(i + 1, n)]

        for (ci, cj) in pairs:
            resolvents = resolve(ci, cj)
            for resolvent in resolvents:
                if not resolvent: # Found the empty clause (contradiction)
                    print(f"\n[Step {step}] Resolved {ci} and {cj} -> EMPTY CLAUSE (Contradiction Found!)")
                    return True
                if frozenset(resolvent) not in [frozenset(c) for c in clauses] and frozenset(resolvent) not in new:
                    print(f"[Step {step}] Resolved {ci} and {cj} -> {resolvent}")
                    new.add(frozenset(resolvent))
                    step += 1

        if new.issubset({frozenset(c) for c in clauses}):
            print("\nNo new clauses can be resolved. KB does not entail the query.")
            return False

        for c in new:
            clauses.append(set(c))
        new.clear()

In [4]:
# Example usage of the resolution solver:
# Let KB be: (P or Q), (~P or R), (~Q or R)
# Query: R
kb = [
    {'P', 'Q'},
    {'~P', 'R'},
    {'~Q', 'R'}
]
query = 'R'

entails = pl_resolution(kb, query)
print(f"\nDoes the KB entail '{query}'? {entails}")

Initial Clauses:
 C1: {'Q', 'P'}
 C2: {'R', '~P'}
 C3: {'~Q', 'R'}
 C4: {'~R'}
[Step 1] Resolved {'Q', 'P'} and {'R', '~P'} -> {'Q', 'R'}
[Step 2] Resolved {'Q', 'P'} and {'~Q', 'R'} -> {'P', 'R'}
[Step 3] Resolved {'R', '~P'} and {'~R'} -> {'~P'}
[Step 4] Resolved {'~Q', 'R'} and {'~R'} -> {'~Q'}
[Step 5] Resolved {'Q', 'P'} and {'~P'} -> {'Q'}
[Step 6] Resolved {'Q', 'P'} and {'~Q'} -> {'P'}
[Step 7] Resolved {'R', '~P'} and {'P', 'R'} -> {'R'}

[Step 8] Resolved {'~R'} and {'R'} -> EMPTY CLAUSE (Contradiction Found!)

Does the KB entail 'R'? True


### Proving "John likes peanuts" using Resolution (Slide 5 & 6 Case Study)

In [5]:
# We model the specific predicates and variables to show the step-by-step resolution.
# For propositional-style resolution matching this domain:
# 1. ~Food(peanuts) or Likes(John, peanuts)
# 2. ~Eats(Anil, peanuts) or Killed(Anil) or Food(peanuts)
# 3. Eats(Anil, peanuts)
# 4. Alive(Anil)
# 5. ~Alive(Anil) or ~Killed(Anil)
# Query to prove: Likes(John, peanuts) -> Negated Query: ~Likes(John, peanuts)

case_study_kb = [
    {'~Food(peanuts)', 'Likes(John, peanuts)'},          # From: John likes all food
    {'~Eats(Anil, peanuts)', 'Killed(Anil)', 'Food(peanuts)'}, # From: Eats and not killed implies food
    {'Eats(Anil, peanuts)'},                             # From: Anil eats peanuts
    {'Alive(Anil)'},                                     # From: Anil is alive
    {'~Alive(Anil)', '~Killed(Anil)'}                    # From: Alive implies not killed
]

query_to_prove = 'Likes(John, peanuts)'

# Run the resolution on the case study
result = pl_resolution(case_study_kb, query_to_prove)
print(f"\nCan we prove '{query_to_prove}' using resolution? {result}")

Initial Clauses:
 C1: {'~Food(peanuts)', 'Likes(John, peanuts)'}
 C2: {'Killed(Anil)', 'Food(peanuts)', '~Eats(Anil, peanuts)'}
 C3: {'Eats(Anil, peanuts)'}
 C4: {'Alive(Anil)'}
 C5: {'~Killed(Anil)', '~Alive(Anil)'}
 C6: {'~Likes(John, peanuts)'}
[Step 1] Resolved {'~Food(peanuts)', 'Likes(John, peanuts)'} and {'Killed(Anil)', 'Food(peanuts)', '~Eats(Anil, peanuts)'} -> {'Killed(Anil)', 'Likes(John, peanuts)', '~Eats(Anil, peanuts)'}
[Step 2] Resolved {'~Food(peanuts)', 'Likes(John, peanuts)'} and {'~Likes(John, peanuts)'} -> {'~Food(peanuts)'}
[Step 3] Resolved {'Killed(Anil)', 'Food(peanuts)', '~Eats(Anil, peanuts)'} and {'Eats(Anil, peanuts)'} -> {'Killed(Anil)', 'Food(peanuts)'}
[Step 4] Resolved {'Killed(Anil)', 'Food(peanuts)', '~Eats(Anil, peanuts)'} and {'~Killed(Anil)', '~Alive(Anil)'} -> {'~Alive(Anil)', 'Food(peanuts)', '~Eats(Anil, peanuts)'}
[Step 5] Resolved {'Alive(Anil)'} and {'~Killed(Anil)', '~Alive(Anil)'} -> {'~Killed(Anil)'}
[Step 6] Resolved {'~Food(peanuts)', 'L